# Study 2: Reward design and pollution pressure

- Earlier analyses suggested that agents often stop developing while still able to afford more growth. I want to see whether paying for ongoing income changes that pattern under the current environment.
- I compare three economic rewards: the current EDI gain, income above the starting income, and both together. Local, global, and mixed sharing still work as before.
- First I hold the current environment fixed. Later sections examine pollutant transport and test weaker high prevention and cleanup settings.
- This notebook keeps policies and tables in memory. It does not write CSV, JSON, or image files.- The saved tables and notes below are from the latest notebook run. Read each reward comparison alongside its pollution and spending measures.

## Settings

In [1]:
import pandas as pd

from study_2_economic_sustainability.train import train_agents
from study_2_economic_sustainability.evaluation import run_episode
from study_2_economic_sustainability.config import EnvironmentConfig, TrainingConfig
from study_2_economic_sustainability.environment import (EconomicSustainabilityEnvironment, action_parts)

In [2]:
REWARD_DESIGNS = ("edi_gain", "income", "combined")
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))
TRAINING_EPISODES = 2_000
INCOME_REWARD_SCALE = 10.0
environment_config = EnvironmentConfig()
training_config = TrainingConfig()
training_config.training_episodes = TRAINING_EPISODES

## Define the economic rewards

- `edi_gain` isolates the immediate reward for EDI gained on the current step.
- `income` gives `(income earned this step - base income) / 10`. It pays 0 at EDI 0 and 5 at EDI 100. I subtract base income so every action does not receive an unrelated survival bonus.
- `combined` adds both economic terms. The scale of 10 puts the income bonus near the current middle development gain of 5 at EDI 100.

In [3]:
def economic_reward(info, region, design, config):
    edi_gain = info["development_gains"][region]
    income_bonus = (info["income"][region] - config.income_base) / INCOME_REWARD_SCALE

    if design == "income":
        return income_bonus
    if design == "combined":
        return edi_gain + income_bonus
    return edi_gain

In [4]:
def learning_rewards(environment, info, terminated, design):
    config = environment.config
    local_rewards = []

    for region in range(4):
        reward = config.economy_weight * economic_reward(info, region, design, config)
        reward -= config.pollution_weight * info["pollution_damage"][region]
        if not info["valid_actions"][region]:
            reward -= config.invalid_action_penalty
        if terminated:
            reward -= config.failure_penalty
        local_rewards.append(reward)

    shared_reward = sum(local_rewards) / 4

    if environment.reward_mode == "global":
        return (shared_reward,) * 4
    if environment.reward_mode == "mixed":
        mixed_rewards = []
        for reward in local_rewards:
            mixed_rewards.append(0.7 * reward + 0.3 * shared_reward)
        return tuple(mixed_rewards)
    return tuple(local_rewards)

## Train matched policies

In [5]:
def train_policy(design, reward_mode, training_seed, config):
    def reward_transform(environment, rewards, info, terminated):
        return learning_rewards(environment, info, terminated, design)

    return train_agents(reward_mode, training_seed, config, training_config, reward_transform)

In [6]:
trained_policies = {}

for design in REWARD_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            trained_policies[(design, reward_mode, training_seed)] = train_policy(design, reward_mode, training_seed, environment_config)
        print(f"Trained {design} with {reward_mode} rewards across five seeds")

Trained edi_gain with local rewards across five seeds
Trained edi_gain with global rewards across five seeds
Trained edi_gain with mixed rewards across five seeds
Trained income with local rewards across five seeds
Trained income with global rewards across five seeds
Trained income with mixed rewards across five seeds
Trained combined with local rewards across five seeds
Trained combined with global rewards across five seeds
Trained combined with mixed rewards across five seeds


## Evaluate the same starts

- GPI is the average global pollution index over an episode.
- `peak_regional_gpi` is the highest regional pollution index reached during an episode. `high_investment_percent` counts selected high prevention or high cleanup actions. Spending includes valid prevention and cleanup purchases.

In [7]:
def empty_totals():
    return {
        "gpi": 0.0, "spillover": 0.0, "peak_regional_gpi": 0.0,
        "mitigation_spending": 0, "development_actions": 0, "high_actions": 0,
        "last_development_steps": [0, 0, 0, 0],
    }

In [8]:
def add_step_totals(totals, actions, info, step):
    totals["gpi"] += info["global_pollution_index"]
    for pollutant in info["incoming_emissions"]:
        for region in range(4):
            totals["spillover"] += info["incoming_emissions"][pollutant][region]
            totals["spillover"] += info["incoming_pollution"][pollutant][region]

    totals["peak_regional_gpi"] = max(totals["peak_regional_gpi"], *info["regional_pollution_indices"])
    for region in range(4):
        totals["mitigation_spending"] += info["prevention_costs_paid"][region]
        totals["mitigation_spending"] += info["cleanup_costs_paid"][region]

        if info["development_gains"][region] > 0:
            totals["development_actions"] += 1
            totals["last_development_steps"][region] = step

        investment_choice = action_parts(actions[region])[1]
        totals["high_actions"] += investment_choice in (3, 6)

In [9]:
def episode_result(environment, totals, terminated):
    region_steps = 4 * environment.step_count
    return {
        "steps": environment.step_count,
        "failure_percent": 100 * terminated,
        "mean_gpi": totals["gpi"] / environment.step_count,
        "peak_regional_gpi": totals["peak_regional_gpi"],
        "mitigation_spending": totals["mitigation_spending"],
        "final_edi": sum(environment.development_indices) / 4,
        "final_currency": sum(environment.currency_balances) / 4,
        "development_percent": 100 * totals["development_actions"] / region_steps,
        "last_development_step": sum(totals["last_development_steps"]) / 4,
        "high_investment_percent": 100 * totals["high_actions"] / region_steps,
        "mean_incoming_spillover": totals["spillover"] / region_steps,
    }

In [10]:
def evaluate_episode(agents, reward_mode, seed, config):
    environment = EconomicSustainabilityEnvironment(reward_mode, config)
    totals = empty_totals()
    transitions, terminated, truncated = run_episode(environment, agents, seed)

    for transition in transitions:
        add_step_totals(totals, transition["actions"], transition["info"], transition["step"])
    return episode_result(environment, totals, terminated)

In [11]:
episode_rows = []

for design in REWARD_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            agents = trained_policies[(design, reward_mode, training_seed)]
            for environment_seed in EVALUATION_SEEDS:
                row = evaluate_episode(agents, reward_mode, environment_seed, environment_config)
                row.update(reward_design=design, reward_mode=reward_mode, training_seed=training_seed)
                episode_rows.append(row)

episodes = pd.DataFrame(episode_rows)
print(f"Evaluated {len(episodes)} episodes")

Evaluated 900 episodes


## Compare outcomes

- `last_development_step` averages each region's final successful development step. A region that never develops contributes 0. This helps show whether investment continues later than in notebook 07.
- More development is not automatically better. I want to see if extra EDI is earned without making pollution, spending, or failure unacceptable for this synthetic study.

In [ ]:
MEASURES = [
    "mean_gpi", "peak_regional_gpi", "mitigation_spending",
    "final_edi", "final_currency", "failure_percent",
    "development_percent", "last_development_step",
    "high_investment_percent",
    "mean_incoming_spillover",
]

In [12]:
seed_results = episodes.groupby(["reward_design", "reward_mode", "training_seed"])[MEASURES].mean()
mean_results = seed_results.groupby(level=["reward_design", "reward_mode"]).mean()
seed_spread = seed_results.groupby(level=["reward_design", "reward_mode"]).std()

In [13]:
display(mean_results[["final_edi", "development_percent", "last_development_step", "final_currency", "mitigation_spending"]].round(1))

final_edi  development_percent  \
reward_design reward_mode                                   
combined      global            73.2                  5.8   
              local             79.0                  7.5   
              mixed             81.9                  8.0   
edi_gain      global            54.0                  5.3   
              local             71.5                  5.4   
              mixed             68.0                  5.1   
income        global            69.2                  4.9   
              local             70.0                  5.0   
              mixed             70.0                  5.0   

                           last_development_step  final_currency  \
reward_design reward_mode                                          
combined      global                        26.0          5718.8   
              local                         32.7          4955.9   
              mixed                         35.5          5847.4   
edi_gain      global                        33.7          2846.8   
              local                         21.1           512.7   
              mixed                         24.8           439.5   
income        global                        24.4          4657.4   
              local                         20.7          1381.8   
              mixed                         20.8          2923.1   

                           mitigation_spending  
reward_design reward_mode                       
combined      global                   18507.2  
              local                    22151.0  
              mixed                    19173.8  
edi_gain      global                   21758.2  
              local                    39224.7  
              mixed                    38076.0  
income        global                   21946.4  
              local                    35545.8  
              mixed                    29368.0

- The numerical notes below summarize the saved comparison.
- Compare final EDI and the last development step with final currency and environmental spending. More currency alone does not show that a reward produced a better balance.

In [14]:
display(mean_results[["mean_gpi", "peak_regional_gpi", "failure_percent", "high_investment_percent"]].round(1))

mean_gpi  peak_regional_gpi  failure_percent  \
reward_design reward_mode                                                 
combined      global           14.9               22.6              0.0   
              local            12.4               21.4              0.0   
              mixed            16.4               24.6              0.0   
edi_gain      global            5.7               15.7              0.0   
              local             2.9               15.4              0.0   
              mixed             2.7               13.6              0.0   
income        global           10.6               18.0              0.0   
              local             4.8               17.5              0.0   
              mixed             5.0               17.8              0.0   

                           high_investment_percent  
reward_design reward_mode                           
combined      global                          18.7  
              local                           22.6  
              mixed                           20.9  
edi_gain      global                          17.1  
              local                           54.8  
              mixed                           52.4  
income        global                          29.6  
              local                           57.8  
              mixed                           30.7

- The numerical notes below summarize the saved results for the current design.
- Compare the economic outcomes with GPI, pollutant loads, spending, and failures. High investment now includes prevention and cleanup choices.

In [15]:
display(seed_spread[["final_edi", "mean_gpi", "mitigation_spending", "failure_percent"]].round(1))

final_edi  mean_gpi  mitigation_spending  \
reward_design reward_mode                                             
combined      global             3.2       5.1               2771.6   
              local             12.9       5.9               5289.9   
              mixed              8.2       2.5               4236.2   
edi_gain      global            31.0       2.9               6924.6   
              local              3.4       0.5               1779.7   
              mixed              4.5       0.4               2128.8   
income        global             2.5       2.4               3479.0   
              local              0.0       1.6               3616.9   
              mixed              0.1       1.3               5786.5   

                           failure_percent  
reward_design reward_mode                   
combined      global                   0.0  
              local                    0.0  
              mixed                    0.0  
edi_gain      global                   0.0  
              local                    0.0  
              mixed                    0.0  
income        global                   0.0  
              local                    0.0  
              mixed                    0.0

- The table reports variation across training seeds. Large spread means the result depends more on which seed trained the policy.
- Compare spread across training seeds as well as the mean; a reward mode may behave differently across independent training runs.

## How do regions affect each other?

- The current environment transports each pollutant's remaining emissions and recovered pollution stock between neighboring regions. The transport share differs by pollutant, and transferred amounts leave the source.
- The table below illustrates the immediate effect of five extra air-emission units before transport. Region 0 keeps the untransported share; regions 1 and 2 each receive an equal share of the transported amount. Region 3 receives none directly.

In [ ]:
EXTRA_AIR_EMISSIONS = 5
AIR_TRANSPORT_FRACTION = environment_config.transport_fractions["air"]
spillover_rows = []

In [16]:
for source_region in range(4):
    row = {"source_region": source_region}
    for affected_region in range(4):
        if source_region == affected_region:
            change = EXTRA_AIR_EMISSIONS * (1 - AIR_TRANSPORT_FRACTION)
        elif source_region in environment_config.neighbors[affected_region]:
            change = EXTRA_AIR_EMISSIONS * AIR_TRANSPORT_FRACTION
            change /= len(environment_config.neighbors[source_region])
        else:
            change = 0
        row[f"region_{affected_region}_load_change"] = change
    spillover_rows.append(row)

display(pd.DataFrame(spillover_rows).set_index("source_region"))

,region_0_load_change,region_1_load_change,region_2_load_change,region_3_load_change
source_region,,,,
0,3.50,0.75,0.75,0.00
1,0.75,3.50,0.00,0.75
2,0.75,0.00,3.50,0.75
3,0.00,0.75,0.75,3.50


### Spillover under the learned policies

- `mean_incoming_spillover` combines transported emissions and transported pollution stocks across the four pollutants. It is the average inflow per region-step, resulting from the joint policies and pollutant-specific transport settings.

In [17]:
display(mean_results[["mean_incoming_spillover"]].round(2))

mean_incoming_spillover
reward_design reward_mode                         
combined      global                         13.32
              local                          12.19
              mixed                          14.84
edi_gain      global                          7.03
              local                           6.17
              mixed                           5.79
income        global                         10.38
              local                           6.73
              mixed                           7.48

- Read regional pollution alongside transport, natural recovery, and each region's own investment choices.
- After rerunning, compare incoming transport with regional stocks and GPI. A region's own mitigation and natural recovery also affect its pollution load.

## What if high environmental investment is weaker?

- This stress test reduces only the strongest prevention fraction and cleanup capacity. Costs and the other investment levels stay fixed.
- It checks whether the highest environmental investment options have enough effect to justify their cost under the current pollutant-specific dynamics.

In [18]:
stressed_config = EnvironmentConfig()
stressed_config.prevention_fractions = (0.0, 0.25, 0.50, 0.60)
stressed_config.cleanup_capacities = (0.0, 2.0, 5.0, 8.0)
investment_rows = []

for name, settings in (("current", environment_config), ("weaker_high", stressed_config)):
    investment_rows.append({
        "setting": name,
        "high_prevention_fraction": settings.prevention_fractions[3],
        "high_cleanup_capacity_per_pollutant": settings.cleanup_capacities[3],
    })

display(pd.DataFrame(investment_rows).set_index("setting"))

,high_prevention_fraction,high_cleanup_capacity_per_pollutant
setting,,
current,0.75,10.0
weaker_high,0.60,8.0


### Run the weaker-investment test after reviewing the reward results

- Set `RUN_STRESS` to `True` to train policies under the weaker high-investment settings. This adds 45 training runs.
- The comparison subtracts the current-setting result from the weaker-high result for each design, reward mode, and training seed.

In [19]:
RUN_STRESS = True

In [20]:
stressed_policies = {}
if RUN_STRESS:
    for design in REWARD_DESIGNS:
        for reward_mode in REWARD_MODES:
            for training_seed in TRAINING_SEEDS:
                stressed_policies[(design, reward_mode, training_seed)] = train_policy(
                    design, reward_mode, training_seed, stressed_config
                )
            print(f"Trained weaker-high {design}, {reward_mode} across five seeds")

Trained weaker-high edi_gain, local across five seeds
Trained weaker-high edi_gain, global across five seeds
Trained weaker-high edi_gain, mixed across five seeds
Trained weaker-high income, local across five seeds
Trained weaker-high income, global across five seeds
Trained weaker-high income, mixed across five seeds
Trained weaker-high combined, local across five seeds
Trained weaker-high combined, global across five seeds
Trained weaker-high combined, mixed across five seeds


In [21]:
stressed_rows = []
if RUN_STRESS:
    for design in REWARD_DESIGNS:
        for reward_mode in REWARD_MODES:
            for training_seed in TRAINING_SEEDS:
                agents = stressed_policies[(design, reward_mode, training_seed)]
                for environment_seed in EVALUATION_SEEDS:
                    row = evaluate_episode(agents, reward_mode, environment_seed, stressed_config)
                    row.update(reward_design=design, reward_mode=reward_mode, training_seed=training_seed)
                    stressed_rows.append(row)
    stressed_episodes = pd.DataFrame(stressed_rows)
    print(f"Evaluated {len(stressed_episodes)} weaker-high episodes")

Evaluated 900 weaker-high episodes


In [22]:
if RUN_STRESS:
    stressed_seed_results = stressed_episodes.groupby(["reward_design", "reward_mode", "training_seed"])[MEASURES].mean()
    investment_change = stressed_seed_results - seed_results
    investment_change_mean = investment_change.groupby(level=["reward_design", "reward_mode"]).mean()

    display(investment_change_mean[[
        "mean_gpi", "peak_regional_gpi", "failure_percent",
        "final_edi", "last_development_step", "mitigation_spending",
        "high_investment_percent",
        "mean_incoming_spillover",]].round(1))

mean_gpi  peak_regional_gpi  failure_percent  \
reward_design reward_mode                                                 
combined      global           -1.9               -3.2              0.0   
              local             0.3               -0.9              0.0   
              mixed             0.2                0.8              0.0   
edi_gain      global           -0.3               -1.8              0.0   
              local            -0.0               -0.8              0.0   
              mixed             1.6                2.3              0.0   
income        global           -0.4               -0.3              0.0   
              local             4.1                2.2              0.0   
              mixed             0.8               -2.0              0.0   

                           final_edi  last_development_step  \
reward_design reward_mode                                     
combined      global            -1.5                   -1.6   
              local             -3.9                   -6.4   
              mixed             -2.2                   -2.0   
edi_gain      global             1.0                   -3.8   
              local              0.2                    4.8   
              mixed              0.0                   -3.2   
income        global             0.8                    1.0   
              local              0.2                   -0.7   
              mixed             -0.0                    1.0   

                           mitigation_spending  high_investment_percent  \
reward_design reward_mode                                                 
combined      global                     778.8                      3.3   
              local                      -27.3                      3.5   
              mixed                     1762.4                     -2.7   
edi_gain      global                    4851.6                     12.9   
              local                      811.2                     -6.6   
              mixed                    -1342.2                     -4.2   
income        global                    2091.4                     -0.7   
              local                    -9831.0                    -34.2   
              mixed                    -3546.6                    -15.8   

                           mean_incoming_spillover  
reward_design reward_mode                           
combined      global                          -0.5  
              local                            0.2  
              mixed                           -0.3  
edi_gain      global                          -0.3  
              local                            0.2  
              mixed                            1.0  
income        global                           0.2  
              local                            3.5  
              mixed                            0.4

- Compare pollution, development, spending, failures, and action choices together. Retraining can change policy choices, so weaker investment does not guarantee higher pollution.
- Compare changes in pollution, development, spending, failures, and chosen investment levels. Retraining may change policy choices, so weaker investment does not guarantee higher GPI.

## Does the weaker-investment result hold across seeds?

- The first table shows each combined-reward seed. The second shows the standard deviation of paired differences for all reward designs. Positive differences mean weaker high investment increased the measure.

In [23]:
display(investment_change.loc["combined", ["mean_gpi", "final_edi", "mitigation_spending", "high_investment_percent"]].round(1))

mean_gpi  final_edi  mitigation_spending  \
reward_mode training_seed                                             
global      11                 -0.4        0.8               2954.0   
            23                  5.3       -7.5               -868.0   
            37                 -6.6        4.5               7586.2   
            41                -11.3       -5.4              -5499.8   
            53                  3.3       -0.1               -278.5   
local       11                 -2.9       -9.8              -8044.8   
            23                 -3.7      -17.2               8875.5   
            37                 11.2       17.7              -7413.2   
            41                 -0.2        0.0               3936.8   
            53                 -2.9      -10.1               2509.2   
mixed       11                  5.5        5.0                315.5   
            23                  6.5        3.4               3120.8   
            37                 -7.5       -0.1               6077.2   
            41                 -6.8      -10.4               7218.2   
            53                  3.4       -8.7              -7920.0   

                           high_investment_percent  
reward_mode training_seed                           
global      11                                 7.5  
            23                                17.7  
            37                                19.6  
            41                               -34.6  
            53                                 6.4  
local       11                               -37.8  
            23                                34.2  
            37                               -11.6  
            41                                27.2  
            53                                 5.3  
mixed       11                                -4.5  
            23                                16.8  
            37                               -17.1  
            41                                31.4  
            53                               -39.9

In [24]:
investment_seed_spread = investment_change.groupby(level=["reward_design", "reward_mode"])[["mean_gpi", "final_edi","mitigation_spending"]].std()

display(investment_seed_spread.round(1))

mean_gpi  final_edi  mitigation_spending
reward_design reward_mode                                          
combined      global            6.9        4.9               4856.0
              local             6.2       13.5               7420.2
              mixed             6.8        7.0               6043.0
edi_gain      global            4.8       41.4               9445.8
              local             0.6        5.0               2069.0
              mixed             1.9       10.8               8058.8
income        global            5.8        2.9               6621.7
              local             7.2        0.2              11024.6
              mixed             1.9        0.1               9679.2

## Which regions bear the pollution?

- I compare each region under the current and weaker-high settings. `remaining_emissions` sums emissions after prevention across the four pollutants.
- `incoming_spillover` sums incoming emissions and transported pollution stocks across pollutants. Regional stocks also reflect natural recovery and cleanup.

In [25]:
def empty_region_totals():
    return {
        "pollution_load": [0.0] * 4,
        "remaining_emissions": [0.0] * 4,
        "incoming_spillover": [0.0] * 4,
    }

In [26]:
def add_region_step(totals, info):
    for region in range(4):
        for pollutant in info["pollutant_loads"]:
            totals["pollution_load"][region] += info["pollutant_loads"][pollutant][region]
            totals["remaining_emissions"][region] += info["remaining_emissions"][pollutant][region]
            totals["incoming_spillover"][region] += info["incoming_emissions"][pollutant][region]
            totals["incoming_spillover"][region] += info["incoming_pollution"][pollutant][region]

In [27]:
def evaluate_regions(agents, reward_mode, seed, config):
    environment = EconomicSustainabilityEnvironment(reward_mode, config)
    totals = empty_region_totals()
    transitions, terminated, truncated = run_episode(environment, agents, seed)
    for transition in transitions:
        add_region_step(totals, transition["info"])

    rows = []
    for region in range(4):
        rows.append({
            "region": region,
            "mean_pollution_load": totals["pollution_load"][region] / environment.step_count,
            "mean_remaining_emissions": totals["remaining_emissions"][region] / environment.step_count,
            "mean_incoming_spillover": totals["incoming_spillover"][region] / environment.step_count,
            "final_edi": environment.development_indices[region],
        })
    return rows

In [28]:
region_rows = []
if RUN_STRESS:
    for setting, config, policies in (
        ("current", environment_config, trained_policies),
        ("weaker_high", stressed_config, stressed_policies)):
        for design in REWARD_DESIGNS:
            for reward_mode in REWARD_MODES:
                for training_seed in TRAINING_SEEDS:
                    agents = policies[(design, reward_mode, training_seed)]
                    for environment_seed in EVALUATION_SEEDS:
                        rows = evaluate_regions(agents, reward_mode, environment_seed, config)
                        for row in rows:
                            row.update(setting=setting, design=design, reward_mode=reward_mode, training_seed=training_seed)
                        region_rows.extend(rows)

    regional_episodes = pd.DataFrame(region_rows)
    print(f"Recorded {len(regional_episodes)} region-episodes")

Recorded 7200 region-episodes


In [29]:
if RUN_STRESS:
    REGION_MEASURES = ["mean_pollution_load", "mean_remaining_emissions", "mean_incoming_spillover", "final_edi"]
    region_seed = regional_episodes.groupby(["setting", "design", "reward_mode", "training_seed", "region"])[REGION_MEASURES].mean()
    region_change = region_seed.loc["weaker_high"] - region_seed.loc["current"]
    region_mean_change = region_change.groupby(level=["design", "reward_mode", "region"]).mean()
    display(region_mean_change.loc["combined"].round(2))

mean_pollution_load  mean_remaining_emissions  \
reward_mode region                                                  
global      0                    -11.40                     -1.14   
            1                    -11.53                      3.10   
            2                      3.61                     -1.28   
            3                    -12.67                      1.79   
local       0                     -2.04                     -0.02   
            1                      8.49                      1.37   
            2                     -0.89                     -0.31   
            3                     -0.86                     -0.49   
mixed       0                     -4.55                      5.20   
            1                     -0.25                     -0.08   
            2                      6.60                     -3.89   
            3                      2.56                     -1.23   

                    mean_incoming_spillover  final_edi  
reward_mode region                                      
global      0                          0.15      -3.72  
            1                         -1.18       0.00  
            2                         -1.18       0.00  
            3                          0.15      -2.47  
local       0                          0.60      -6.22  
            1                         -0.28       0.32  
            2                         -0.28      -0.25  
            3                          0.60      -9.28  
mixed       0                         -0.49       3.44  
            1                         -0.08       2.66  
            2                         -0.08     -18.72  
            3                         -0.49       3.98

## What if development produces more emissions?

- This section compares the current development-related emissions rates with rates that are 50% larger. Baseline emissions remain unchanged, and all four pollutants are included.
- This is a synthetic sensitivity, not a calibrated estimate. Interpret it as a comparison within this model, not as a real-world estimate.
- I retrain the current EDI-gain and combined rewards for all three sharing modes, using the same five training seeds and 20 evaluation starts. This adds 30 training runs. I leave income-only aside because it did not consistently extend development in the first comparison.

In [ ]:
emissions_config = EnvironmentConfig()

for pollutant in emissions_config.pollutants:
    emissions_config.edi_emissions_per_point[pollutant] *= 1.5

SLOPE_DESIGNS = ("edi_gain", "combined")
emissions_preview = []

In [30]:
for edi in (0, 70, 100):
    emissions_preview.append({"edi": edi, "current_total_emissions": 0, "higher_rate_total_emissions": 0})
    for pollutant in environment_config.pollutants:
        current_emissions = environment_config.baseline_emissions[pollutant]
        current_emissions += environment_config.edi_emissions_per_point[pollutant] * edi
        higher_emissions = emissions_config.baseline_emissions[pollutant]
        higher_emissions += emissions_config.edi_emissions_per_point[pollutant] * edi
        emissions_preview[-1]["current_total_emissions"] += current_emissions
        emissions_preview[-1]["higher_rate_total_emissions"] += higher_emissions

display(pd.DataFrame(emissions_preview).set_index("edi"))

,current_total_emissions,higher_rate_total_emissions
edi,,
0,6.5,6.50
70,18.4,24.35
100,23.5,32.00


### Train with higher development-related emissions

In [31]:
emissions_policies = {}
for design in SLOPE_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            emissions_policies[(design, reward_mode, training_seed)] = train_policy(
                design, reward_mode, training_seed, emissions_config
            )
        print(f"Trained higher-emissions {design}, {reward_mode} across five seeds")

Trained higher-emissions edi_gain, local across five seeds
Trained higher-emissions edi_gain, global across five seeds
Trained higher-emissions edi_gain, mixed across five seeds
Trained higher-emissions combined, local across five seeds
Trained higher-emissions combined, global across five seeds
Trained higher-emissions combined, mixed across five seeds


### Evaluate matched starts

In [32]:
emissions_rows = []
for design in SLOPE_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            agents = emissions_policies[(design, reward_mode, training_seed)]
            for environment_seed in EVALUATION_SEEDS:
                row = evaluate_episode(agents, reward_mode, environment_seed, emissions_config)
                row.update(reward_design=design, reward_mode=reward_mode, training_seed=training_seed)
                emissions_rows.append(row)

emissions_episodes = pd.DataFrame(emissions_rows)
print(f"Evaluated {len(emissions_episodes)} higher-rate episodes")

Evaluated 600 higher-rate episodes


In [33]:
emissions_seed_results = emissions_episodes.groupby(["reward_design", "reward_mode", "training_seed"])[MEASURES].mean()
base_slope_seed = seed_results[seed_results.index.get_level_values("reward_design").isin(SLOPE_DESIGNS)]

emissions_change = emissions_seed_results - base_slope_seed
emissions_mean = emissions_seed_results.groupby(level=["reward_design", "reward_mode"]).mean()

final_edi  last_development_step  mean_gpi  \
reward_design reward_mode                                               
combined      global            91.2                   66.7      22.5   
              local             90.7                   59.2      21.5   
              mixed             91.1                   55.3      19.5   
edi_gain      global            54.1                   26.3       8.1   
              local             71.5                   23.2       4.4   
              mixed             71.5                   25.2       4.7   

                           peak_regional_gpi  failure_percent  \
reward_design reward_mode                                       
combined      global                    32.9              1.0   
              local                     32.3              4.0   
              mixed                     30.2              0.0   
edi_gain      global                    18.2              0.0   
              local                     19.9              0.0   
              mixed                     19.1              0.0   

                           mitigation_spending  mean_incoming_spillover  
reward_design reward_mode                                                
combined      global                   21751.0                     23.6  
              local                    24140.2                     21.4  
              mixed                    23976.8                     21.4  
edi_gain      global                   25492.2                      9.8  
              local                    40353.4                      8.9  
              mixed                    39380.4                      8.8

In [ ]:
display(emissions_mean[["final_edi", "last_development_step", "mean_gpi", "peak_regional_gpi", "failure_percent", "mitigation_spending", "mean_incoming_spillover",]].round(1))

- The numerical notes below summarize the saved results for the current 50% rate sensitivity.
- Compare development against pollution, mitigation spending, incoming transport, and failures. The pollutant-specific sensitivity may produce a different response.

In [34]:
emissions_change_mean = emissions_change.groupby(level=["reward_design", "reward_mode"]).mean()

display(emissions_change_mean[["final_edi", "mean_gpi", "failure_percent", "mitigation_spending", "high_investment_percent",
    "mean_incoming_spillover"]].round(1))

final_edi  mean_gpi  failure_percent  \
reward_design reward_mode                                         
combined      global            18.0       7.6              1.0   
              local             11.7       9.1              4.0   
              mixed              9.2       3.1              0.0   
edi_gain      global             0.1       2.4              0.0   
              local              0.0       1.5              0.0   
              mixed              3.5       2.1              0.0   

                           mitigation_spending  high_investment_percent  \
reward_design reward_mode                                                 
combined      global                    3243.8                     -0.4   
              local                     1989.2                      7.6   
              mixed                     4803.1                      4.3   
edi_gain      global                    3734.0                     11.6   
              local                     1128.8                    -11.0   
              mixed                     1304.4                     -2.7   

                           mean_incoming_spillover  
reward_design reward_mode                           
combined      global                          10.3  
              local                            9.2  
              mixed                            6.6  
edi_gain      global                           2.8  
              local                            2.7  
              mixed                            3.0

- Each value is the higher-rate outcome minus the current-rate outcome for the same design, reward mode, and training seed.
- Use the paired differences to see whether the weaker high-investment setting changed outcomes consistently. Action frequencies show whether agents shifted toward prevention, cleanup, or development.

In [35]:
emissions_change_spread = emissions_change.groupby(level=["reward_design", "reward_mode"])[["final_edi", "mean_gpi", "mitigation_spending"]].std()

display(emissions_change_spread.round(1))

final_edi  mean_gpi  mitigation_spending
reward_design reward_mode                                          
combined      global             4.2       5.4               3285.1
              local             14.2       6.8               3798.8
              mixed             12.0       2.4               5208.7
edi_gain      global            25.9       3.8              10190.6
              local              5.3       0.7               1810.0
              mixed              4.9       0.3               2874.0

## Which design should we keep?

- This table puts the current and higher-emissions-rate environments beside one another for the two rewards tested in both. Each row averages five training seeds.
- I would look for development that continues when it is useful, mitigation that responds to pollution, manageable spillover and failures, and results that do not depend heavily on one seed.

In [36]:
current_candidates = base_slope_seed.reset_index()
current_candidates["environment"] = "current"
higher_candidates = emissions_seed_results.reset_index()
higher_candidates["environment"] = "higher_edi_emissions"
candidates = pd.concat([current_candidates, higher_candidates], ignore_index=True)
candidate_means = candidates.groupby(["environment", "reward_design", "reward_mode"])[MEASURES].mean()

final_edi  \
environment          reward_design reward_mode              
current              combined      global            73.2   
                                   local             79.0   
                                   mixed             81.9   
                     edi_gain      global            54.0   
                                   local             71.5   
                                   mixed             68.0   
higher_edi_emissions combined      global            91.2   
                                   local             90.7   
                                   mixed             91.1   
                     edi_gain      global            54.1   
                                   local             71.5   
                                   mixed             71.5   

                                                last_development_step  \
environment          reward_design reward_mode                          
current              combined      global                        26.0   
                                   local                         32.7   
                                   mixed                         35.5   
                     edi_gain      global                        33.7   
                                   local                         21.1   
                                   mixed                         24.8   
higher_edi_emissions combined      global                        66.7   
                                   local                         59.2   
                                   mixed                         55.3   
                     edi_gain      global                        26.3   
                                   local                         23.2   
                                   mixed                         25.2   

                                                mean_gpi  failure_percent  \
environment          reward_design reward_mode                              
current              combined      global           14.9              0.0   
                                   local            12.4              0.0   
                                   mixed            16.4              0.0   
                     edi_gain      global            5.7              0.0   
                                   local             2.9              0.0   
                                   mixed             2.7              0.0   
higher_edi_emissions combined      global           22.5              1.0   
                                   local            21.5              4.0   
                                   mixed            19.5              0.0   
                     edi_gain      global            8.1              0.0   
                                   local             4.4              0.0   
                                   mixed             4.7              0.0   

                                                mitigation_spending  \
environment          reward_design reward_mode                        
current              combined      global                   18507.2   
                                   local                    22151.0   
                                   mixed                    19173.8   
                     edi_gain      global                   21758.2   
                                   local                    39224.7   
                                   mixed                    38076.0   
higher_edi_emissions combined      global                   21751.0   
                                   local                    24140.2   
                                   mixed                    23976.8   
                     edi_gain      global                   25492.2   
                                   local                    40353.4   
                                   mixed                    39380.4   

                                                mean_incoming_spillover  
environment          r

In [ ]:
display(candidate_means[[
    "final_edi", "last_development_step", "mean_gpi",
    "failure_percent", "mitigation_spending",
    "mean_incoming_spillover"]].round(1))

In [37]:
CONFIRMATION_DESIGNS = ("edi_gain", "combined")
CONFIRMATION_TRAINING_SEEDS = tuple(range(101, 121))
CONFIRMATION_EVALUATION_SEEDS = tuple(range(200_000, 200_020))

print(
    f"{len(CONFIRMATION_TRAINING_SEEDS)} new training seeds and "
    f"{len(CONFIRMATION_EVALUATION_SEEDS)} new evaluation starts"
)

20 new training seeds and 20 new evaluation starts


### Train the confirmation policies

In [38]:
confirmation_policies = {}
for design in CONFIRMATION_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in CONFIRMATION_TRAINING_SEEDS:
            confirmation_policies[(design, reward_mode, training_seed)] = train_policy(
                design, reward_mode, training_seed, environment_config
            )
        print(f"Trained confirmation {design}, {reward_mode} across 20 seeds")

Trained confirmation edi_gain, local across 20 seeds
Trained confirmation edi_gain, global across 20 seeds
Trained confirmation edi_gain, mixed across 20 seeds
Trained confirmation combined, local across 20 seeds
Trained confirmation combined, global across 20 seeds
Trained confirmation combined, mixed across 20 seeds


### Evaluate fresh starts

In [39]:
confirmation_rows = []
for design in CONFIRMATION_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in CONFIRMATION_TRAINING_SEEDS:
            agents = confirmation_policies[(design, reward_mode, training_seed)]
            for environment_seed in CONFIRMATION_EVALUATION_SEEDS:
                row = evaluate_episode(agents, reward_mode, environment_seed, environment_config)

                row.update(reward_design=design, reward_mode=reward_mode, training_seed=training_seed, environment_seed=environment_seed)

                confirmation_rows.append(row)

confirmation_episodes = pd.DataFrame(confirmation_rows)
print(f"Evaluated {len(confirmation_episodes)} confirmation episodes")

Evaluated 2400 confirmation episodes


### Compare outcomes and seed variation

- Each training seed is summarized over the same 20 fresh evaluation starts. The first table averages those seed summaries; the second shows their standard deviation.
- These are physical outcomes and action shares. I do not compare average rewards because the economic reward formulas have different scales.

In [40]:
confirmation_seed = confirmation_episodes.groupby(["reward_design", "reward_mode", "training_seed"])[MEASURES].mean()
confirmation_mean = confirmation_seed.groupby(level=["reward_design", "reward_mode"]).mean()

final_edi  last_development_step  mean_gpi  \
reward_design reward_mode                                               
combined      global            73.0                   28.1      13.6   
              local             78.6                   31.2      10.9   
              mixed             77.3                   30.8      13.6   
edi_gain      global            64.5                   35.6       6.8   
              local             70.9                   24.1       2.6   
              mixed             68.7                   25.9       3.1   

                           peak_regional_gpi  failure_percent  \
reward_design reward_mode                                       
combined      global                    21.7              5.0   
              local                     21.3              0.0   
              mixed                     22.7              0.0   
edi_gain      global                    15.1              0.0   
              local                     15.1              0.0   
              mixed                     15.3              0.0   

                           mitigation_spending  high_investment_percent  \
reward_design reward_mode                                                 
combined      global                   19966.0                     21.9   
              local                    24657.2                     26.1   
              mixed                    21424.6                     22.4   
edi_gain      global                   26572.7                     36.5   
              local                    39798.1                     52.3   
              mixed                    36902.2                     49.9   

                           mean_incoming_spillover  
reward_design reward_mode                           
combined      global                          12.3  
              local                           11.3  
              mixed                           12.8  
edi_gain      global                           7.8  
              local                            6.1  
              mixed                            6.3

In [ ]:
display(confirmation_mean[[
    "final_edi", "last_development_step", "mean_gpi",
    "peak_regional_gpi", "failure_percent",
    "mitigation_spending", "high_investment_percent",
    "mean_incoming_spillover"]].round(1))

In [41]:
confirmation_spread = confirmation_seed.groupby(level=["reward_design", "reward_mode"])[["final_edi", "mean_gpi", "mitigation_spending", "failure_percent"]].std()

display(confirmation_spread.round(1))

final_edi  mean_gpi  mitigation_spending  \
reward_design reward_mode                                             
combined      global             5.7       5.3               5094.9   
              local              8.1       6.5               7597.8   
              mixed              7.5       6.7               5656.8   
edi_gain      global            19.2       3.5               6054.8   
              local              3.3       0.4                557.9   
              mixed             10.6       0.6               5415.0   

                           failure_percent  
reward_design reward_mode                   
combined      global                  22.4  
              local                    0.0  
              mixed                    0.0  
edi_gain      global                   0.0  
              local                    0.0  
              mixed                    0.0

In [42]:
confirmation_difference = (confirmation_seed.loc["combined"] - confirmation_seed.loc["edi_gain"])
difference_mean = confirmation_difference.groupby(level="reward_mode").mean()
difference_spread = confirmation_difference.groupby(level="reward_mode").std()

In [43]:
display(difference_mean[["final_edi", "mean_gpi", "failure_percent","mitigation_spending", "high_investment_percent", "mean_incoming_spillover"]].round(1))

,final_edi,mean_gpi,failure_percent,mitigation_spending,high_investment_percent,mean_incoming_spillover
reward_mode,,,,,,
global,8.5,6.8,5.0,-6606.8,-14.6,4.5
local,7.7,8.2,0.0,-15140.9,-26.3,5.2
mixed,8.6,10.5,0.0,-15477.7,-27.6,6.5


In [44]:
display(difference_spread[["final_edi", "mean_gpi","mitigation_spending"]].round(1))


,final_edi,mean_gpi,mitigation_spending
reward_mode,,,
global,19.4,5.6,5600.4
local,8.0,6.6,7772.4
mixed,13.0,6.8,7536.0


In [45]:
confirmation_direction = confirmation_difference.assign(more_edi=confirmation_difference["final_edi"] > 0, more_gpi=confirmation_difference["mean_gpi"] > 0)

direction_percent = (confirmation_direction.groupby(level="reward_mode")[["more_edi", "more_gpi"]].mean() * 100)
display(direction_percent.round(1))

,more_edi,more_gpi
reward_mode,,
global,55.0,90.0
local,75.0,100.0
mixed,65.0,100.0


In [46]:
confirmation_region_rows = []
for design in CONFIRMATION_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in CONFIRMATION_TRAINING_SEEDS:
            agents = confirmation_policies[(design, reward_mode, training_seed)]
            for environment_seed in CONFIRMATION_EVALUATION_SEEDS:
                rows = evaluate_regions(agents, reward_mode, environment_seed, environment_config)
                for row in rows:
                    row.update(reward_design=design, reward_mode=reward_mode, training_seed=training_seed)
                confirmation_region_rows.extend(rows)

confirmation_regions = pd.DataFrame(confirmation_region_rows)
print(f"Recorded {len(confirmation_regions)} region-episodes")

Recorded 9600 region-episodes


In [47]:
CONFIRMATION_REGION_MEASURES = ["mean_pollution_load", "mean_remaining_emissions", "mean_incoming_spillover", "final_edi"]
confirmation_region_seed = confirmation_regions.groupby(["reward_design", "reward_mode", "training_seed", "region"])[CONFIRMATION_REGION_MEASURES].mean()

mean_pollution_load  mean_remaining_emissions  \
reward_mode region                                                  
global      0                     28.92                      2.90   
            1                     28.89                     -0.67   
            2                     21.07                      0.72   
            3                     21.97                      0.53   
local       0                     32.09                     -1.56   
            1                     33.45                      0.19   
            2                     25.61                     -1.54   
            3                     29.03                      1.02   
mixed       0                     37.36                      0.61   
            1                     35.53                     -0.67   
            2                     42.49                     -0.91   
            3                     38.84                     -2.65   

                    mean_incoming_spillover  final_edi  
reward_mode region                                      
global      0                          4.28      21.69  
            1                          4.81      -5.48  
            2                          4.81       8.32  
            3                          4.28       9.28  
local       0                          5.11       7.62  
            1                          5.38       5.88  
            2                          5.38       6.83  
            3                          5.11      10.61  
mixed       0                          6.55       9.02  
            1                          6.38       8.22  
            2                          6.38       7.18  
            3                          6.55       9.80

In [ ]:
region_difference = (confirmation_region_seed.loc["combined"] - confirmation_region_seed.loc["edi_gain"])
display(region_difference.groupby(level=["reward_mode", "region"]).mean().round(2))

## Working reward decision for the current model

## What does high prevention leave behind?

- The environment tracks four separate regional pollution stocks. It does not use one shared atmospheric CO2 stock, so a single airborne fraction would not describe its current dynamics.
- This table shows only how high prevention changes new emissions at several EDI levels. Transport, natural recovery, and cleanup then affect the pollution stocks separately.

In [48]:
EDI_LEVELS = (0, 70, 100)
HIGH_PREVENTION_LEVEL = 3

### Gross and remaining emissions

- The calculation compares gross emissions with remaining emissions after high prevention.

In [49]:
emissions_rows = []

for edi in EDI_LEVELS:
    gross_total = 0
    remaining_total = 0
    for pollutant in environment_config.pollutants:
        gross = environment_config.baseline_emissions[pollutant]
        gross += environment_config.edi_emissions_per_point[pollutant] * edi
        remaining = gross * (1 - environment_config.prevention_fractions[HIGH_PREVENTION_LEVEL])
        gross_total += gross
        remaining_total += remaining

    emissions_rows.append({
        "edi": edi,
        "gross_emissions": gross_total,
        "remaining_after_high_prevention": remaining_total,
    })

display(pd.DataFrame(emissions_rows).set_index("edi"))

,gross_emissions,remaining_after_high_prevention
edi,,
0,6.5,1.625
70,18.4,4.600
100,23.5,5.875


In [50]:
cleanup_rows = []

for pollutant in environment_config.pollutants:
    cleanup_rows.append({
        "pollutant": pollutant,
        "high_cleanup_capacity_per_pollutant": environment_config.cleanup_capacities[3],
        "high_cleanup_cost": environment_config.cleanup_costs[3],
        "cleanup_difficulty": environment_config.cleanup_difficulty[pollutant],
        "natural_recovery_rate": environment_config.natural_recovery_rates[pollutant],
    })

display(pd.DataFrame(cleanup_rows).set_index("pollutant"))

,high_cleanup_capacity_per_pollutant,high_cleanup_cost,cleanup_difficulty,natural_recovery_rate
pollutant,,,,
air,10.0,80.0,0.00,0.60
water,10.0,80.0,0.50,0.12
soil,10.0,80.0,0.65,0.02
noise,10.0,80.0,0.00,0.90
